<a href="https://colab.research.google.com/github/ubiodee/plutustutor/blob/main/Copy_of_Cardano_PlutusLearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [ ]:

!pip uninstall -y bitsandbytes -q
!pip install -U -q "transformers>=4.44.2" "accelerate>=0.34.2" "datasets>=2.20.0" \
                    "sentencepiece>=0.2.0" "huggingface_hub>=0.24.6" \
                    "bitsandbytes>=0.43.1" "peft>=0.12.0"

import os, gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Login with your token stored in Colab secrets
login(token=userdata.get("HF_TOKEN"))


dataset = load_dataset("json", data_files={"train": "/content/TRAINING_DATA_PLUTUS.json"})
model_name = "meta-llama/Llama-3.2-3B-Instruct"

def format_data(example):
    topic = example.get("Topic", "No topic provided")
    level = example.get("Level", "No level provided")
    personality = example.get("Personality", "No personality provided")
    content = example.get("Content", "No content provided")
    example["text"] = (
        "### Instruction:\n"
        "You are an AI Plutus tutor. Explain Plutus concepts clearly and adapt them to the learner’s needs.\n\n"
        f"### Topic:\n{topic}\n\n"
        f"### Level:\n{level}\n\n"
        f"### Personality Type:\n{personality}\n\n"
        f"### Content:\n{content}"
    )
    return example

dataset = dataset["train"].map(format_data)


tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

def tokenize_function(example):
    tok = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    tok["labels"] = tok["input_ids"].clone()
    tok["labels"][tok["labels"] == tokenizer.pad_token_id] = -100
    return tok

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized.set_format("torch")


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
)

# Make sure new pad token is recognized
model.resize_token_embeddings(len(tokenizer))

# Enable memory savers for training
model.gradient_checkpointing_enable()
model.config.use_cache = False  # required when gradient checkpointing is on

# Prepare k-bit model (casts norms etc. for stable training)
model = prepare_model_for_kbit_training(model)


lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Good defaults for LLaMA-family; adjust if needed:
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,     # effective batch size = 8
    num_train_epochs=3,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="no",
    logging_steps=10,
    logging_dir="./logs",
    report_to="none",
    push_to_hub=True,
    hub_model_id="ubiodee/Plutuslearn-Llama-3.2-3B-Instruct",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,                         # compute in FP16
    optim="paged_adamw_8bit",          # memory-efficient optimizer (bitsandbytes)
)


trainer = Trainer(model=model, args=training_args, train_dataset=tokenized)
trainer.train()


adapter_dir = "./plutus_lora_adapter"
model.save_pretrained(adapter_dir)                # saves LoRA adapters (small)
tokenizer.save_pretrained("./tokenizer")

# Push to Hub (adapters repo)
from huggingface_hub import create_repo
create_repo("ubiodee/Plutuslearn-Llama-3.2-3B-Instruct", exist_ok=True)
trainer.push_to_hub()                             # pushes adapter weights + training card
tokenizer.push_to_hub("ubiodee/Plutuslearn-Llama-3.2-3B-Instruct")


Map:   0%|          | 0/607 [00:00<?, ? examples/s]

Map:   0%|          | 0/607 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 12,156,928 || all params: 3,224,909,824 || trainable%: 0.3770


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmp_fmqdw57/tokenizer.json       : 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/ubiodee/Plutuslearn-Llama-3.2-3B-Instruct/commit/365c9c0d05fd4f7bb071ec53ae0ed4beefd24ac7', commit_message='Upload tokenizer', commit_description='', oid='365c9c0d05fd4f7bb071ec53ae0ed4beefd24ac7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ubiodee/Plutuslearn-Llama-3.2-3B-Instruct', endpoint='https://huggingface.co', repo_type='model', repo_id='ubiodee/Plutuslearn-Llama-3.2-3B-Instruct'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import pipeline
# In ipython-input-2-4a157b5a023a:
prompt = "Write a PlutusTx script that always succeeds and passes validation."

# Create a text generation pipeline using your fine-tuned model
pipe = pipeline(
    "text-generation",
    model="./results",  # Path to your fine-tuned model
    tokenizer=tokenizer,
    # ... other pipeline parameters if needed ...
)


NameError: name 'tokenizer' is not defined